# 09C – REST API for Bankruptcy Risk Prediction (Enterprise Edition)

Production-ready FastAPI notebook scaffold.

In [ ]:
import joblib
import pandas as pd
import numpy as np
from typing import List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH='../models/production_bankruptcy_model.joblib'
DATA_PATH='../data/processed/american_bankruptcy_cleaned.csv'

model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)
TARGET_COLUMN='Bankrupt?'
feature_columns=[c for c in df.columns if c!=TARGET_COLUMN]

app=FastAPI(title='Enterprise Bankruptcy Prediction API')

@app.get('/')
def root():
    return {'status':'running'}

@app.get('/health')
def health():
    return {'status':'healthy'}

class PredictionRequest(BaseModel):
    features: List[float]

@app.post('/predict')
def predict(req: PredictionRequest):
    if len(req.features)!=len(feature_columns):
        raise HTTPException(status_code=400,detail='Incorrect feature count')
    x=np.array(req.features).reshape(1,-1)
    pred=int(model.predict(x)[0])
    prob=float(model.predict_proba(x)[0][1]) if hasattr(model,'predict_proba') else float(pred)
    return {'prediction':pred,'probability':prob,'risk':'High Risk' if prob>=0.5 else 'Low Risk'}
